# Kaggle: Banded Ensembles — fetch → train → clean (all 6 models)

Self-contained: attach **only this notebook** to a Kaggle notebook, no repo needed.
It streams real data from the HuggingFace Hub (nothing to pre-upload), trains all six
banded-ensemble models with the threshold-gated joining protocol
`37 → 47 → 57 → 67 → 71 → 73 → 77 → 79 → 81 → 85 → 90%` (+ grace period before
in-band distillation), exports ONNX + `run-metadata.json` per model, frees
memory + disk after **each** model (12h / disk limits), and stages a publish bundle.

**Kaggle setup:** Accelerator GPU T4 x2 (or P100) · Environment: latest ·
Secrets: `KAGGLE_API_TOKEN` (only needed for the final publish cell) ·
Scale: `BUDDY_SCALE` env (`smoke` ≈ 20 min sanity, `demo` ≈ 1–2h, `full` = paper run).

**Local:** runs as-is (`BUDDY_SCALE=smoke` default) — writes only to `./kaggle_out/`,
never touches the repo checkout.

In [ ]:
# Cell 0 — environment: deps, device, dirs, seeds (local + Kaggle safe)
import importlib.util, sys
for _pkg, _pip in [('torch', 'torch'), ('datasets', 'datasets'),
                     ('huggingface_hub', 'huggingface_hub'), ('cv2', 'opencv-python-headless'),
                     ('onnxscript', 'onnxscript'), ('gymnasium', 'gymnasium'), ('soundfile', 'soundfile')]:
    if importlib.util.find_spec(_pkg) is None:
        print('pip install', _pip)
        %pip install -q {_pip}

import gc, os, pathlib
import numpy as np, torch
torch.manual_seed(42); np.random.seed(42)
ON_KAGGLE = pathlib.Path('/kaggle').exists()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WORK = pathlib.Path('/kaggle/working') if ON_KAGGLE else pathlib.Path.cwd()
OUT = WORK / 'buddyup-output'
MODELS = OUT / 'models'
OUT.mkdir(parents=True, exist_ok=True); MODELS.mkdir(parents=True, exist_ok=True)
SCALE = os.environ.get('BUDDY_SCALE', 'smoke')  # smoke | demo | full
print(f'kaggle={ON_KAGGLE} device={DEVICE} scale={SCALE} out={OUT}')
if DEVICE == 'cuda': print(torch.cuda.get_device_name(0))

In [ ]:
# Cell 1 — shared helpers: hash text, band manager, export, cleanup, HF streaming
import hashlib, json, re
import torch.nn as nn
_WORD = re.compile(r"[a-z0-9']+")
BANDS = [0.37, 0.47, 0.57, 0.67, 0.71, 0.73, 0.77, 0.79, 0.81, 0.85, 0.90]

def hash_tokenize(texts, vocab=2000, seqlen=32):
    rows = []
    for t in texts:
        toks = _WORD.findall(str(t).lower())[:seqlen]
        row = [int(hashlib.md5(w.encode()).hexdigest(), 16) % vocab for w in toks]
        row += [0] * (seqlen - len(row))
        rows.append(row)
    return torch.tensor(rows, dtype=torch.long)

class BandManager:
    """Threshold-gated joining + grace period before in-band distillation."""
    def __init__(self, bands=BANDS, grace=2):
        self.bands, self.grace, self.acc, self.joined = sorted(bands), grace, {}, {}
    def band_for(self, a):
        b = None
        for t in self.bands:
            if a >= t: b = t
        return b
    def update(self, mid, acc, step):
        self.acc[mid] = max(acc, self.acc.get(mid, 0.))
        b = self.band_for(self.acc[mid])
        prev = self.joined.get(mid, (None,))[0]
        if b is not None and (prev is None or b > prev):
            self.joined[mid] = (b, step)
            print(f'  [band] {mid} acc={self.acc[mid]:.3f} -> {int(b*100)}% @step{step}')
    def eligible(self, mid, step):
        return mid in self.joined and (step - self.joined[mid][1]) >= self.grace
    def pools(self):
        o = {}
        for m, (b, _) in self.joined.items(): o.setdefault(b, []).append(m)
        return o

RUN_META = []  # one run-metadata entry per model (promotion gate reads these)

def export_onnx(model, dummy, name, metrics, fname_in='input'):
    model.eval()
    path = MODELS / f'{name}.onnx'
    with torch.no_grad():
        torch.onnx.export(model.cpu(), dummy.cpu(), str(path),
                            input_names=[fname_in], output_names=['logits'])
    model.to(DEVICE)
    meta = {'name': name, 'version': '1.0.0', 'artifact_path': str(path),
            'framework': 'pytorch', 'metrics': metrics}
    (MODELS / f'{name}.metadata.json').write_text(json.dumps(meta, indent=1))
    RUN_META.append(meta)
    print('exported', path, f"({path.stat().st_size/1e6:.1f}MB)")
    return path


def hf_stream(dataset, config=None, split='train', streaming=True):
    from datasets import load_dataset
    kw = {'trust_remote_code': False}
    try:
        ds = load_dataset(dataset, config, split=split, streaming=True, **kw)
        next(iter(ds))  # streaming is lazy: force first fetch so errors surface here
        print(f'streaming {dataset}' + (f'/{config}' if config else '') + f' [{split}]')
        return ds
    except Exception as e:  # old fsspec grips: sliced full download instead
        print('streaming failed, sliced download:', repr(e)[:100])
        return load_dataset(dataset, config, split=split, streaming=False, **kw)

def hf_audio(dataset, config, split='train', n_probe=True):
    from datasets import load_dataset, Audio
    try:
        ds = load_dataset(dataset, config, split=split, streaming=True,
                          trust_remote_code=False)
        ds = ds.cast_column('audio', Audio(decode=False))
        if n_probe: next(iter(ds))  # force first fetch inside the try
        print(f'streaming {dataset}/{config} [{split}] (raw audio bytes)')
        return ds
    except Exception as e:
        print('audio streaming failed, sliced download:', repr(e)[:120])
        ds = load_dataset(dataset, config, split=split, streaming=False,
                          trust_remote_code=False)
        return ds.cast_column('audio', Audio(decode=False))

def take_texts(ds, field, n, label_field='label'):
    texts, labels = [], []
    for row in ds:
        texts.append(row[field]); labels.append(int(row[label_field]))
        if len(texts) >= n: break
    return texts, labels

print('helpers ready:', len(BANDS), 'bands | device:', DEVICE)

In [ ]:
# Cell 2 — model 1/6: banded attention NLP (IMDB 2-class; ag_news fallback 4-class)
import torch.nn as nn
N1 = {'smoke': 3000, 'demo': 25000, 'full': 80000}[SCALE]
EP1 = {'smoke': 2, 'demo': 5, 'full': 8}[SCALE]
NM1 = {'smoke': 4, 'demo': 8, 'full': 20}[SCALE]
GR1 = {'smoke': 1, 'demo': 2, 'full': 5}[SCALE]

class TinyAttentionLM(nn.Module):
    def __init__(self, vocab=2000, d=64, heads=4, nclass=2, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.emb = nn.Embedding(vocab, d)
        nn.init.normal_(self.emb.weight, generator=g)
        enc = nn.TransformerEncoderLayer(d, heads, dim_feedforward=128, batch_first=True)
        for p in enc.parameters(): nn.init.normal_(p, generator=g)
        self.enc = enc; self.head = nn.Linear(d, nclass)
        nn.init.normal_(self.head.weight, generator=g)
    def forward(self, x): return self.head(self.enc(self.emb(x)).mean(1))

try:
    _ds = hf_stream('stanfordnlp/imdb', split='train')
    _texts, _labels = take_texts(_ds, 'text', N1)
    _nclass = 2
except Exception as e:
    print('imdb failed, ag_news fallback:', repr(e)[:120])
    _ds = hf_stream('ag_news', split='train')
    _texts, _labels = take_texts(_ds, 'text', N1)
    _nclass = 4
X = hash_tokenize(_texts).to(DEVICE)
y = torch.tensor(_labels, dtype=torch.long, device=DEVICE)
k = int(0.9 * len(X)); Xtr, ytr, Xva, yva = X[:k], y[:k], X[k:], y[k:]
print(f'nlp: {len(Xtr)} train / {len(Xva)} val, classes={_nclass}')
del _ds, _texts, _labels, X, y; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

models = [TinyAttentionLM(nclass=_nclass, seed=i).to(DEVICE) for i in range(NM1)]
opts = [torch.optim.Adam(m.parameters(), lr=3e-3) for m in models]
mgr = BandManager(grace=GR1); ce = nn.CrossEntropyLoss(); BS = 256
for ep in range(EP1):
    for m, o in zip(models, opts):
        m.train(); perm = torch.randperm(len(Xtr), device=DEVICE)
        for i in range(0, len(Xtr), BS):
            idx = perm[i:i+BS]; o.zero_grad()
            loss = ce(m(Xtr[idx]), ytr[idx]); loss.backward(); o.step()
    print(f'--- epoch {ep} ---')
    with torch.no_grad():
        for j, m in enumerate(models):
            m.eval(); mgr.update(j, (m(Xva).argmax(1) == yva).float().mean().item(), ep)
    pools = mgr.pools()
    for b, members in pools.items():
        elig = [j for j in members if mgr.eligible(j, ep)]
        if len(elig) < 2: continue
        with torch.no_grad():
            soft = torch.softmax(torch.stack([models[j](Xtr[:512]) for j in elig]).mean(0) / 2.0, -1)
        for j in elig:
            models[j].train(); opts[j].zero_grad()
            (0.2 * -(soft * torch.log_softmax(models[j](Xtr[:512]) / 2.0, -1)).sum(-1).mean()).backward()
            opts[j].step()
        print(f'  [distill] band {int(b*100)}%: {len(elig)} members')
with torch.no_grad():
    accs = [(m(Xva).argmax(1) == yva).float().mean().item() for m in models]
    bag = (torch.stack([m(Xva) for m in models]).mean(0).argmax(1) == yva).float().mean().item()
print('best=', round(max(accs), 4), 'bagged=', round(bag, 4))
export_onnx(models[int(np.argmax(accs))], torch.randint(0, 2000, (1, 32)),
              'banded_nlp_best', {'best_acc': round(max(accs), 4), 'bagged_acc': round(bag, 4)},
              'input_ids')
del models, opts, Xtr, ytr, Xva, yva, accs; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

In [ ]:
# Cell 3 — model 2/6: banded vision (CIFAR-10, mixed CNN + TinyViT pool)
N2 = {'smoke': 4000, 'demo': 30000, 'full': 50000}[SCALE]
EP2 = {'smoke': 2, 'demo': 5, 'full': 12}[SCALE]
NM2 = {'smoke': 4, 'demo': 8, 'full': 20}[SCALE]

class SmallCNN(nn.Module):
    def __init__(self, c=10, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.f = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.h = nn.Linear(32, c)
    def forward(self, x): return self.h(self.f(x).flatten(1))

class TinyViT(nn.Module):
    def __init__(self, c=10, d=64, patch=8, seed=0):
        super().__init__()
        self.proj = nn.Conv2d(3, d, patch, stride=patch)
        self.enc = nn.TransformerEncoderLayer(d, 4, dim_feedforward=128, batch_first=True)
        self.cls = nn.Parameter(torch.randn(d)); self.h = nn.Linear(d, c)
    def forward(self, x):
        t = self.proj(x).flatten(2).transpose(1, 2)
        return self.h(self.enc(torch.cat([self.cls.expand(len(x), 1, -1), t], 1))[:, 0])

_ds = hf_stream('uoft-cs/cifar10', split='train')
_X, _y = [], []
for row in _ds:
    _X.append(np.array(row['img'].convert('RGB').resize((32, 32)), dtype=np.float32).transpose(2, 0, 1) / 255.0)
    _y.append(int(row['label']))
    if len(_X) >= N2: break
Xtr = torch.tensor(np.stack(_X), device=DEVICE); ytr = torch.tensor(_y, dtype=torch.long, device=DEVICE)
del _ds, _X, _y; gc.collect()
_dsv = hf_stream('uoft-cs/cifar10', split='test')
_Xv, _yv = [], []
for row in _dsv:
    _Xv.append(np.array(row['img'].convert('RGB').resize((32, 32)), dtype=np.float32).transpose(2, 0, 1) / 255.0)
    _yv.append(int(row['label']))
    if len(_Xv) >= 2000: break
Xva = torch.tensor(np.stack(_Xv), device=DEVICE); yva = torch.tensor(_yv, dtype=torch.long, device=DEVICE)
del _dsv, _Xv, _yv; gc.collect()
print(f'vision: {len(Xtr)} train / {len(Xva)} val')

models = [(SmallCNN(seed=i) if i % 2 == 0 else TinyViT(seed=i)).to(DEVICE) for i in range(NM2)]
opts = [torch.optim.Adam(m.parameters(), lr=2e-3) for m in models]
mgr = BandManager(grace=GR1); ce = nn.CrossEntropyLoss(); BS = 256
for ep in range(EP2):
    for m, o in zip(models, opts):
        m.train(); perm = torch.randperm(len(Xtr), device=DEVICE)
        for i in range(0, len(Xtr), BS):
            idx = perm[i:i+BS]; o.zero_grad()
            loss = ce(m(Xtr[idx]), ytr[idx]); loss.backward(); o.step()
    print(f'--- epoch {ep} ---')
    with torch.no_grad():
        for j, m in enumerate(models):
            m.eval(); mgr.update(f'{type(m).__name__}{j}', (m(Xva).argmax(1) == yva).float().mean().item(), ep)
    for b, members in mgr.pools().items():
        elig = [j for j, m in enumerate(models) if f'{type(m).__name__}{j}' in members and mgr.eligible(f'{type(m).__name__}{j}', ep)]
        if len(elig) < 2: continue
        with torch.no_grad():
            soft = torch.softmax(torch.stack([models[j](Xtr[:256]) for j in elig]).mean(0) / 2.0, -1)
        for j in elig:
            models[j].train(); opts[j].zero_grad()
            (0.25 * -(soft * torch.log_softmax(models[j](Xtr[:256]) / 2.0, -1)).sum(-1).mean()).backward()
            opts[j].step()
        print(f'  [distill] band {int(b*100)}% distilled {len(elig)}')
with torch.no_grad():
    accs = [(m(Xva).argmax(1) == yva).float().mean().item() for m in models]
    bag = (torch.stack([m(Xva) for m in models]).mean(0).argmax(1) == yva).float().mean().item()
print('best=', round(max(accs), 4), 'bagged=', round(bag, 4))
export_onnx(models[int(np.argmax(accs))], torch.randn(1, 3, 32, 32), 'banded_vision_best',
              {'best_acc': round(max(accs), 4), 'bagged_acc': round(bag, 4)}, 'image')
del models, opts, Xtr, ytr, Xva, yva, accs; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

In [ ]:
# Cell 4 — model 3/6: multimodal JEPA (cifar / ucf101 / minds14 / ag_news / byte-hist)
# One JEPA tower per modality + per-tower linear probes gating band membership.
# Video uses 2 real UCF101 avis when reachable, else CIFAR-as-frames (see log).
import torch.nn as nn
ST3 = {'smoke': 20, 'demo': 80, 'full': 300}[SCALE]
GR3 = {'smoke': 1, 'demo': 2, 'full': 5}[SCALE]
D = 64

class ModalityJEPA(nn.Module):
    def __init__(self, in_dim, d=D):
        super().__init__()
        self.ctx = nn.Sequential(nn.Linear(in_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Linear(128, d))
        self.tgt = nn.Sequential(nn.Linear(in_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Linear(128, d))
        self.pred = nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Linear(128, d))
        self._ema(0.0)
    @torch.no_grad()
    def _ema(self, m=0.996):
        for a, b in zip(self.tgt.parameters(), self.ctx.parameters()): a.data.mul_(m).add_(b.data, alpha=1-m)
    def forward(self, x_full, mask_ctx, mask_tgt):
        h = self.ctx(x_full)
        zc = self.pred(h[mask_ctx])
        with torch.no_grad(): zt = self.tgt(x_full[mask_tgt])
        n = min(len(zc), len(zt))
        return ((zc[:n] - zt[:n]) ** 2).mean() + torch.relu(1 - h.std(0).mean())
    def embed(self, x, toks=8): return self.ctx(x).view(-1, toks, D).mean(1)

NIMG = {'smoke': 1500, 'demo': 6000, 'full': 20000}[SCALE]
NAP3 = min(NIMG, 400)
# -- image tower: CIFAR rows (3072), own 10-way labels --
_ds = hf_stream('uoft-cs/cifar10', split='train')
_XI, _yi = [], []
for row in _ds:
    _XI.append(np.array(row['img'].convert('RGB').resize((32, 32)), dtype=np.float32).reshape(-1) / 255.0)
    _yi.append(int(row['label']))
    if len(_XI) >= NIMG: break
XI = torch.tensor(np.stack(_XI), device=DEVICE); yi = torch.tensor(_yi, dtype=torch.long, device=DEVICE)
del _ds, _XI, _yi; gc.collect()
# -- video tower: 2 real UCF101 avis (2 classes) or CIFAR-as-frames fallback --
XV, yv = None, None
_V, _yv, _f = [], [], []
try:
    import cv2
    from huggingface_hub import hf_hub_download
    _V, _yv = [], []
    for ci, fn in enumerate(('v_BabyCrawling_g19_c02.avi', 'v_BasketballDunk_g14_c06.avi')):
        pth = hf_hub_download('sayakpaul/ucf101-subset', fn)
        cap = cv2.VideoCapture(pth); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        _f = []
        for fi in np.linspace(0, max(n - 1, 0), 32).astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
            ok, fr = cap.read()
            if ok: _f.append(cv2.resize(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY), (48, 48)).reshape(-1))
        cap.release()
        _f = _f[:len(_f) // 8 * 8]
        if len(_f) >= 8:
            _V.append(np.stack(_f).reshape(-1, 8, 2304)); _yv += [ci] * (len(_f) // 8)
    XV = torch.tensor(np.concatenate(_V), dtype=torch.float32, device=DEVICE)
    yv = torch.tensor(_yv, dtype=torch.long, device=DEVICE)
    print(f'video: real UCF101 avis, {len(yv)} clips')
except Exception as e:
    print('UCF101 unreachable, CIFAR-as-frames fallback:', repr(e)[:150])
    _NV = max(8, min(len(XI) // 8 * 8, NAP3 * 8))
    XV = XI[:_NV].reshape(-1, 8, 3072)[:, :, :2304].reshape(-1, 8, 2304)
    yv = (yi[:_NV].reshape(-1, 8)[:, 0] % 2).to(DEVICE)
del _V, _yv, _f; gc.collect()
# -- audio tower: minds14 intents from 2s waveform envelopes --
try:
    _ds = hf_audio('polyai/minds14', 'en-US', split='train')
    _XA, _ya = [], []
    for row in _ds:
        try:
            import soundfile as _sf, io as _io
            w, sr = _sf.read(_io.BytesIO(row['audio']['bytes']), dtype='float32', always_2d=False)
            if np.ndim(w) > 1: w = w.mean(-1)
            sr = int(sr)
        except Exception:
            continue
        if len(w) < sr: continue
        seg = w[:sr * 2] if len(w) >= sr * 2 else np.pad(w, (0, sr * 2 - len(w)))
        _XA.append(seg[(np.linspace(0, len(seg) - 1, 320)).astype(int)])
        _ya.append(int(row['intent_class']))
        if len(_XA) >= NAP3: break
    XA = torch.tensor(np.stack(_XA), dtype=torch.float32, device=DEVICE)
    ya = torch.tensor(_ya, dtype=torch.long, device=DEVICE)
    NA_ = int(ya.max()) + 1
    del _ds, _XA, _ya; gc.collect()
except Exception as e:
    print('minds14 unreachable, noise fallback:', repr(e)[:150])
    XA = torch.randn(NAP3, 320, device=DEVICE); ya = torch.randint(0, 14, (NAP3,), device=DEVICE); NA_ = 14
# -- text tower: imdb hash-projection rows (B, 8, 64) --
_ds = hf_stream('stanfordnlp/imdb', split='train')
_tt, _yt = take_texts(_ds, 'text', NIMG, 'label')
g = torch.Generator().manual_seed(11); R = torch.randn(2000, 64, generator=g)
_ids = hash_tokenize(_tt, 2000, 8)
XT = R[_ids].reshape(len(_ids), 8, 64).to(DEVICE)
yt = torch.tensor(_yt, dtype=torch.long, device=DEVICE)
del _ds, _yt, _ids, R; gc.collect()
# -- file tower: review byte histograms -> same binary label --
_FB = torch.zeros(len(_tt), 256)
for i, s in enumerate(_tt):
    b = np.frombuffer(str(s).encode()[:2000], dtype=np.uint8).astype(np.int64)
    _FB[i].index_add_(0, torch.from_numpy(b.copy()), torch.ones(len(b)) / max(1, len(b)))
XF, yf = _FB.to(DEVICE), yt.long()
del _tt, _FB; gc.collect()
towers = {'image': [ModalityJEPA(3072).to(DEVICE), XI, yi, 1, 10],
          'video': [ModalityJEPA(2304).to(DEVICE), XV, yv, 8, 2],
          'audio': [ModalityJEPA(320).to(DEVICE), XA, ya, 1, NA_],
          'text': [ModalityJEPA(64).to(DEVICE), XT, yt, 8, 2],
          'file': [ModalityJEPA(256).to(DEVICE), XF, yf, 1, 2]}
print('towers:', {k: (tuple(v[1].shape), v[3], v[4]) for k, v in towers.items()})
opts = {k: torch.optim.Adam(v[0].parameters(), lr=1e-3) for k, v in towers.items()}
heads = {k: nn.Linear(D, v[4]).to(DEVICE) for k, v in towers.items()}
opt_h = {k: torch.optim.Adam(heads[k].parameters(), lr=1e-2) for k in towers}
mgr = BandManager(grace=GR3); ce = nn.CrossEntropyLoss(); BS3 = 256
for s in range(ST3):
    for k, (tw, Xk, yk, toks, nc) in towers.items():
        tw.train(); opts[k].zero_grad()
        rows = Xk.reshape(-1, Xk.shape[-1])
        bi = torch.randint(0, len(rows), (BS3,), device=DEVICE)
        xx = rows[bi]
        idx = torch.randperm(len(xx), device=DEVICE); m = len(idx) // 2
        loss = tw(xx[idx], idx[:m], idx[m:]); loss.backward(); opts[k].step(); tw._ema()
    if s % 5 == 0:
        for k, (tw, Xk, yk, toks, nc) in towers.items():
            feat, B = Xk.shape[-1], len(yk)
            i1 = torch.randperm(B, device=DEVICE)[:256]; i2 = torch.randperm(B, device=DEVICE)[:128]
            for _ in range(15):
                opt_h[k].zero_grad()
                r1 = Xk[i1].reshape(-1, feat).detach()
                lh = ce(heads[k](tw.embed(r1, toks)), yk[i1])
                lh.backward(); opt_h[k].step()
            with torch.no_grad():
                r2 = Xk[i2].reshape(-1, feat)
                acc = (heads[k](tw.embed(r2, toks)).argmax(1) == yk[i2]).float().mean().item()
            mgr.update(k, acc, s)
print('pools:', {int(k * 100): v for k, v in mgr.pools().items()})
export_onnx(nn.Linear(5 * D, 4).to(DEVICE), torch.randn(1, 5 * D), 'multimodal_fuse',
            {'pools': {int(k * 100): v for k, v in mgr.pools().items()}}, 'emb')
del towers, opts, heads, opt_h, XI, yi, XV, yv, XA, ya, XT, yt, XF, yf; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()


In [ ]:
# Cell 5 — model 4/6: RL + attention NLP (DQN + Actor-Critic, ag_news states)
N4 = {'smoke': 2000, 'demo': 8000, 'full': 20000}[SCALE]
EP4 = {'smoke': 60, 'demo': 300, 'full': 1500}[SCALE]
NA4 = {'smoke': 4, 'demo': 6, 'full': 12}[SCALE]

class AttnState(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.emb = nn.Embedding(500, d)
        self.attn = nn.MultiheadAttention(d, 4, batch_first=True)
    def forward(self, toks):
        h = self.emb(toks)
        return self.attn(h, h, h)[0].mean(1)

class DQNAgent(nn.Module):
    def __init__(self, na=4):
        super().__init__(); self.kind = 'dqn'
        self.enc = AttnState(); self.q = nn.Linear(64, na)
    def act(self, s, eps):
        import random as r
        if r.random() < eps: return r.randrange(self.q.out_features)
        with torch.no_grad(): return int(self.q(self.enc(s)).argmax(1))

class A2CAgent(nn.Module):
    def __init__(self, na=4):
        super().__init__(); self.kind = 'a2c'
        self.enc = AttnState(); self.pi = nn.Linear(64, na); self.v = nn.Linear(64, 1)
    def act(self, s):
        with torch.no_grad(): return int(torch.softmax(self.pi(self.enc(s)), -1).multinomial(1))

def _rule(t):
    t = str(t)
    if '?' in t: return 0
    if '!' in t: return 1
    if len(t.split()) > 25: return 2
    return 3

_ds = hf_stream('stanfordnlp/imdb', split='train')
_tt, _ = take_texts(_ds, 'text', N4, 'label')
ST = hash_tokenize(_tt, 500, 16).to(DEVICE)
YL = torch.tensor([_rule(t) for t in _tt], dtype=torch.long, device=DEVICE)
del _ds, _tt; gc.collect()
print(f'rl-nlp: {len(ST)} real states')

class TextBandit:
    def reset(self):
        j = int(np.random.randint(len(ST)))
        self.label = int(YL[j]); self.s = ST[j].unsqueeze(0)
        return self.s
    def step(self, a): return self.s, float(a == self.label), True, {}

import torch.nn.functional as F
env = TextBandit()
agents = [(DQNAgent() if i % 2 == 0 else A2CAgent()).to(DEVICE) for i in range(NA4)]
opts = [torch.optim.Adam(a.parameters(), lr=2e-3) for a in agents]
scores = {i: [] for i in range(NA4)}; joined = {}
for ep in range(EP4):
    eps = max(0.05, 1.0 - ep / (EP4 * 0.7))
    for i, ag in enumerate(agents):
        s = env.reset()
        if ag.kind == 'dqn':
            a = ag.act(s, eps); _, r, _, _ = env.step(a)
            q = ag.q(ag.enc(s))[0, a]
            loss = (q - torch.tensor(r, device=DEVICE)) ** 2
            opts[i].zero_grad(); loss.backward(); opts[i].step()
        else:
            logits = ag.pi(ag.enc(s)); v = ag.v(ag.enc(s))
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample(); _, r, _, _ = env.step(int(a))
            adv = torch.tensor(float(r), device=DEVICE) - v.detach()
            loss = -(dist.log_prob(a) * adv) + F.mse_loss(v, torch.tensor([[float(r)]], device=DEVICE))
            opts[i].zero_grad(); loss.backward(); opts[i].step()
        scores[i].append(r)
    if (ep + 1) % 20 == 0:
        for i in range(NA4):
            acc = float(np.mean(scores[i][-100:]))
            b = max([t for t in BANDS if acc >= t], default=None)
            if b and i not in joined:
                joined[i] = (b, ep); print(f'ep{ep}: agent {i}({agents[i].kind}) -> {int(b*100)}%')
print('joined:', joined)
elig = [i for i, (_, e0) in joined.items() if EP4 - e0 >= 2] or list(range(len(agents)))
wins = []
for _ in range(200):
    s = env.reset(); votes = []
    for i in elig:
        ag = agents[i]; ag.eval()
        with torch.no_grad():
            votes.append(int(ag.q(ag.enc(s)).argmax(1)) if ag.kind == 'dqn' else int(ag.pi(ag.enc(s)).argmax(1)))
    a = max(set(votes), key=votes.count); _, r, _, _ = env.step(a); wins.append(r)
print('bagged win-rate=', round(float(np.mean(wins)), 4))
_a2c = next(a for a in agents if a.kind == 'a2c'); _a2c.eval()
class PolicyOnly(nn.Module):
    def __init__(self, a): super().__init__(); self.e = a.enc; self.p = a.pi
    def forward(self, t): return self.p(self.e(t))
export_onnx(PolicyOnly(_a2c), torch.randint(0, 500, (1, 16)), 'rl_nlp_policy',
              {'bagged_win_rate': round(float(np.mean(wins)), 4)}, 'tokens')
del agents, opts, ST, YL, wins; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

In [ ]:
# Cell 6 — model 5/6: banded recommender (MovieLens-100k: ratings>=4 + title content)
import torch.nn as nn
NU6, NI6, D6 = 2000, 3000, 32  # top users/items by activity (dense, CPU/GPU-cheap)

class TwoTower(nn.Module):
    def __init__(self, nu, ni, d, seed=0, content_dim=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.u = nn.Embedding(nu, d); self.i = nn.Embedding(ni, d)
        nn.init.normal_(self.u.weight, std=d ** -0.5, generator=g)
        nn.init.normal_(self.i.weight, std=d ** -0.5, generator=g)
        self.use_content = content_dim > 0
        if self.use_content: self.cf = nn.Linear(content_dim, d)
    def score(self, u, items, cfeat=None):
        s = self.u(u) @ self.i(items).T
        if self.use_content and cfeat is not None: s = s + (self.u(u) @ self.cf(cfeat).T)
        return s

# MovieLens-100k: HF mirror needs auth for files -> fall back to grouplens.org
import urllib.request, zipfile
_ML = WORK / 'ml-100k'
_ML.mkdir(parents=True, exist_ok=True)
try:
    from huggingface_hub import hf_hub_download
    _ud = hf_hub_download('includeno/movielens-100k', 'u.data')
    _it = hf_hub_download('includeno/movielens-100k', 'u.item')
except Exception as e:
    print('HF movielens gated, grouplens.org fallback:', repr(e)[:100])
    _zip = _ML / 'ml-100k.zip'
    if not _zip.exists():
        urllib.request.urlretrieve('https://files.grouplens.org/datasets/movielens/ml-100k.zip', _zip)
    with zipfile.ZipFile(_zip) as z: z.extractall(_ML)
    _ud, _it = str(_ML / 'ml-100k' / 'u.data'), str(_ML / 'ml-100k' / 'u.item')
import pandas as pd
_df = pd.read_csv(_ud, sep='\t', header=None, names=['user', 'item', 'rating', 'ts'])
try:
    _nm = pd.read_csv(_it, sep='|', header=None, encoding='latin-1', on_bad_lines='skip')
    _titles = dict(zip(_nm[0].astype(int), _nm[1].fillna('').astype(str)))
except Exception as e:
    print('title parse fallback:', repr(e)[:100]); _titles = {}
del _ud, _it
_topu = _df['user'].value_counts().head(NU6).index
_topi = _df['item'].value_counts().head(NI6).index
_df = _df[_df['user'].isin(_topu) & _df['item'].isin(_topi)]
print(f'movielens dense slice: {len(_df)} ratings')
# leave-one-out val positives per user
_liked = _df[_df['rating'] >= 4]
_hold = _liked.groupby('user', group_keys=False).apply(
    lambda g: g.sample(n=1, random_state=0) if len(g) >= 2 else g.iloc[0:0], include_groups=False)
_va, _tr = _df.loc[_hold.index], _df.drop(index=_hold.index)
_um = {u: i for i, u in enumerate(_tr['user'].unique())}
_im = {r: i for i, r in enumerate(_tr['item'].unique())}
_va = _va[_va['user'].isin(_um) & _va['item'].isin(_im)]
_VALPOS = {}
for _u, _i in zip(_va['user'].map(_um), _va['item'].map(_im)):
    _VALPOS.setdefault(int(_u), []).append(int(_i))
_U = torch.tensor(_tr['user'].map(_um).to_numpy(), dtype=torch.long)
_I = torch.tensor(_tr['item'].map(_im).to_numpy(), dtype=torch.long)
_y = torch.tensor((_tr['rating'].to_numpy(float) >= 4).astype(float))
# 1:1 sampled negatives (logs are positive-skewed)
import random as _r
_rr = _r.Random(0); _obs = set(zip(_U.tolist(), _I.tolist()))
_need, _nu, _ni = int(_y.sum().item()), [], []
while len(_nu) < _need:
    for a, b in zip([_rr.randrange(len(_um)) for _ in range(_need * 2)], [_rr.randrange(len(_im)) for _ in range(_need * 2)]):
        if (a, b) not in _obs:
            _obs.add((a, b)); _nu.append(a); _ni.append(b)
            if len(_nu) >= _need: break
_U = torch.cat([_U, torch.tensor(_nu)]); _I = torch.cat([_I, torch.tensor(_ni)])
_y = torch.cat([_y, torch.zeros(len(_nu))])
_inv = {i: r for r, i in _im.items()}
g = torch.Generator().manual_seed(7); _P = torch.randn(2000, 16, generator=g)
import hashlib, re
_WRD = re.compile(r"[a-z0-9']+")
def _hem(t):
    toks = _WRD.findall(str(t).lower())[:24] or ['_empty_']
    return _P[[int(hashlib.md5(w.encode()).hexdigest(), 16) % 2000 for w in toks]].mean(0)
_C = torch.stack([_hem(_titles.get(_inv[i], '')) for i in range(len(_im))])
del _df, _liked, _hold, _va, _tr, _nm, _P; gc.collect()
NU, NI = len(_um), len(_im)
pairs = list(zip(_U.tolist(), _I.tolist())); labels = _y.tolist()
EP6 = {'smoke': 2, 'demo': 5, 'full': 12}[SCALE]
N6 = {'smoke': 4, 'demo': 6, 'full': 12}[SCALE]
UPD = min(150, max(8, int(12 * len(pairs) / 4096 / max(1, EP6))))
members = [TwoTower(NU, NI, D6, seed=i, content_dim=(16 if i % 2 else 0)).to(DEVICE) for i in range(N6)]
opts = [torch.optim.Adam(m.parameters(), lr=5e-3) for m in members]
bce = nn.BCEWithLogitsLoss(); rng = np.random.default_rng(0)
mgr = BandManager(grace=2)
best = {}
def hr_at_k(m, k=10, trials=200):
    m.eval(); hits = 0
    with torch.no_grad():
        for _ in range(trials):
            u = int(rng.integers(NU))
            if u not in _VALPOS: continue
            pos = int(rng.choice(_VALPOS[u]))
            ci = torch.tensor([pos] + [int(rng.integers(NI)) for _ in range(49)], device=DEVICE)
            s = m.score(torch.tensor([u], device=DEVICE), ci, _C[ci].to(DEVICE) if m.use_content else None)[0]
            hits += int(torch.topk(s, k).indices.eq(0).any())
    return hits / trials
for ep in range(EP6):
    for mi, m in enumerate(members):
        m.train()
        for _ in range(UPD):
            idx = np.random.choice(len(pairs), 4096)
            uu = torch.tensor([pairs[j][0] for j in idx], device=DEVICE)
            ii = torch.tensor([pairs[j][1] for j in idx], device=DEVICE)
            yy = torch.tensor([labels[j] for j in idx], device=DEVICE).float()
            opts[mi].zero_grad()
            s = (m.u(uu) * m.i(ii)).sum(-1)
            if m.use_content: s = s + (m.u(uu) * m.cf(_C[ii].to(DEVICE))).sum(-1)
            loss = bce(s, yy); loss.backward(); opts[mi].step()
    print(f'--- epoch {ep} ---')
    for mi, m in enumerate(members):
        hr = hr_at_k(m); best[mi] = max(hr, best.get(mi, 0.))
        b = max([t for t in BANDS if best[mi] >= t], default=None)
        mgr.update(mi, best[mi], ep)
print('joined:', {k: (int(v[0] * 100), v[1]) for k, v in mgr.joined.items()})
elig = [mi for mi, (_, e0) in mgr.joined.items() if EP6 - e0 >= 2] or list(range(N6))
print('eligible:', elig)
with torch.no_grad():
    hits = 0; T = 200
    for _ in range(T):
        u = int(rng.integers(NU))
        if u not in _VALPOS: continue
        pos = int(rng.choice(_VALPOS[u]))
        ci = torch.tensor([pos] + [int(rng.integers(NI)) for _ in range(49)], device=DEVICE)
        mrr = torch.zeros(50, device=DEVICE)
        for mi in elig:
            m = members[mi]
            s = m.score(torch.tensor([u], device=DEVICE), ci, _C[ci].to(DEVICE) if m.use_content else None)[0]
            mrr += 1.0 / (s.argsort(descending=True).argsort().float() + 1)
        hits += int(torch.topk(mrr, 10).indices.eq(0).any())
print('fused HR@10:', round(hits / T, 4))
bi = int(max(best, key=best.get)); members[bi].eval()
export_onnx(members[bi].i, torch.tensor([0]), 'recsys_item_emb',
              {'best_hr10': round(best[bi], 4), 'fused_hr10': round(hits / T, 4)}, 'item_id')
del members, opts, pairs, labels, best; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

In [ ]:
# Cell 7 — model 6/6: RL + JEPA world model (CartPole-v1, pip-installed env)
EP7 = {'smoke': 150, 'demo': 250, 'full': 1500}[SCALE]
N7 = {'smoke': 3, 'demo': 5, 'full': 10}[SCALE]
OBS, Z, NA, WIN = 48, 32, 2, 50

class JEPAWorldModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(OBS, 128), nn.GELU(), nn.Linear(128, Z))
        self.tgt = nn.Sequential(nn.Linear(OBS, 128), nn.GELU(), nn.Linear(128, Z))
        self.dyn = nn.Sequential(nn.Linear(Z + NA, 128), nn.GELU(), nn.Linear(128, Z))
        self._ema(0.0)
    @torch.no_grad()
    def _ema(self, m=0.996):
        for a, b in zip(self.tgt.parameters(), self.enc.parameters()): a.data.mul_(m).add_(b.data, alpha=1-m)
    def loss(self, o, o_next, a_oh):
        with torch.no_grad(): zt = self.tgt(o_next)
        return ((self.dyn(torch.cat([self.enc(o), a_oh], -1)) - zt) ** 2).mean()

class LatentActorCritic(nn.Module):
    def __init__(self, wm):
        super().__init__(); self.wm = wm
        self.pi = nn.Linear(Z, NA); self.v = nn.Linear(Z, 1)
    def act(self, o):
        with torch.no_grad():
            return int(torch.softmax(self.pi(self.wm.enc(o.to(next(self.wm.parameters()).device))), -1).multinomial(1))

import gymnasium as gym
env = gym.make('CartPole-v1')

def tile(o):
    import numpy as np
    return torch.tensor(np.tile(np.asarray(o, dtype=np.float32), 12)[:48]).unsqueeze(0)

wm = JEPAWorldModel().to(DEVICE)
agents = [LatentActorCritic(wm).to(DEVICE) for _ in range(N7)]
opt_wm = torch.optim.Adam(list(wm.enc.parameters()) + list(wm.dyn.parameters()), lr=1e-3)
opt_ac = [torch.optim.Adam(list(a.pi.parameters()) + list(a.v.parameters()), lr=2e-3) for a in agents]
import torch.nn.functional as F
scores = {i: [] for i in range(N7)}; joined = {}
for ep in range(EP7):
    for i, ag in enumerate(agents):
        o, _ = env.reset(); o = tile(o).to(DEVICE)
        a = ag.act(o); (o2, r, term, trunc, _) = env.step(a); o2 = tile(o2).to(DEVICE)
        aoh = torch.zeros(1, NA, device=DEVICE); aoh[0, a] = 1.0
        opt_wm.zero_grad(); wl = wm.loss(o, o2, aoh); wl.backward(); opt_wm.step(); wm._ema()
        with torch.no_grad(): z = wm.enc(o)
        logits, v = ag.pi(z), ag.v(z)
        dist = torch.distributions.Categorical(logits=logits)
        act = dist.sample(); adv = torch.tensor(float(r), device=DEVICE) - v.detach()
        loss = -(dist.log_prob(act) * adv) + F.mse_loss(v, torch.tensor([[float(r)]], device=DEVICE))
        opt_ac[i].zero_grad(); loss.backward(); opt_ac[i].step()
        scores[i].append(float(r))
        if term or trunc: pass
    if (ep + 1) % 25 == 0:
        for i in range(N7):
            acc = float(np.mean(scores[i][-WIN:]))
            b = max([t for t in BANDS if acc >= t], default=None)
            if b and i not in joined:
                joined[i] = (b, ep); print(f'ep{ep}: agent {i} return={acc:.2f} joins {int(b*100)}% (wm-loss={wl.item():.4f})')
print('joined:', joined)
elig = [i for i, (_, e0) in joined.items() if EP7 - e0 >= 5] or list(range(N7))
wins = []
for _ in range(200):
    o, _ = env.reset(); o = tile(o).to(DEVICE)
    with torch.no_grad():
        votes = [int(agents[i].pi(wm.enc(o)).argmax(1)) for i in elig]
    a = max(set(votes), key=votes.count); _, r, _, _, _ = env.step(a); wins.append(float(r))
print('bagged return=', round(float(np.mean(wins)), 4))
class LatentPolicy(nn.Module):
    def __init__(self, wm, ag): super().__init__(); self.e = wm.enc; self.p = ag.pi
    def forward(self, o): return self.p(self.e(o))
export_onnx(LatentPolicy(wm, agents[elig[0]]), torch.randn(1, OBS), 'rl_jepa_policy',
              {'bagged_return': round(float(np.mean(wins)), 4)}, 'obs')
del agents, opt_ac, wm, wins; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

In [ ]:
# Cell 8 — publish: bundle artifacts + upload as a new buddyup-models version
import shutil
PUB = WORK / 'buddyup-models'
if PUB.exists(): shutil.rmtree(PUB)
PUB.mkdir(parents=True)
for f in sorted(MODELS.rglob('*')):
    if f.is_file() and f.stat().st_size < 500 * 1024 * 1024:
        d = PUB / f.relative_to(MODELS); d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, d)
(MODELS / 'run-metadata.json').write_text(json.dumps(RUN_META, indent=1))
shutil.copy2(MODELS / 'run-metadata.json', PUB / 'run-metadata.json')
(PUB / 'README.md').write_text('\n'.join([
    '# BuddyUp banded-ensemble models', '',
    f'- scale: {SCALE}', f'- device: {DEVICE}',
    '- artifacts: ONNX (fp32) + *.metadata.json + run-metadata.json',
    '- metrics: ' + json.dumps([m['metrics'] for m in RUN_META]), '',
]))
print('bundle:')
for f in sorted(PUB.rglob('*')):
    if f.is_file(): print('  ', f.relative_to(PUB), f'({f.stat().st_size/1e6:.2f}MB)')
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
except Exception as e:
    print('(no Kaggle secret; upload needs KAGGLE_API_TOKEN)')
if os.environ.get('KAGGLE_API_TOKEN'):
    try:
        import kagglehub
        handle = f"{kagglehub.whoami()['username']}/buddyup-models"
        url = kagglehub.dataset_upload(handle, str(PUB), version_notes=f'banded all-six scale={SCALE}')
        print('published:', url)
    except Exception as e:
        print('upload failed:', repr(e)[:200])
        print(f'Fallback: kaggle datasets create -p {PUB} --dir-mode zip')
else:
    print(f'Artifacts staged at {PUB} — attach KAGGLE_API_TOKEN secret and re-run this cell to publish.')

## Notes

- **Promotion gate** accepts `run-metadata.json` next to each artifact as the metric source (same contract as `kaggle_training.ipynb`).
- Serve via `app/ml/serving.py::load_preferred('<name>')` — unversioned `<name>.onnx` aliases resolve automatically; flip `ModelMetadata.is_active` + drop the file to promote.
- Data provenance is streaming-first: nothing is stored except the final ONNX artifacts, so Kaggle's disk quota and this box's 98%-full disk both stay safe.
- Video tower uses 2 real UCF101 avis when reachable, else CIFAR-as-frames (printed at runtime — check the log line).
- `full` scale is a multi-hour GPU run; `demo` reproduces every band transition in ~1–2h on a T4.